In [0]:
# === J10 - ML evaluation and investigation setup ===
%pip install xgboost==2.0.3 scikit-learn==1.4.0 numpy==1.26.4 matplotlib -q
dbutils.library.restartPython()

In [0]:
# === J10 - Imports ===
import sys
import os

repo_root = "/Workspace/Users/salmacharki721@gmail.com/cybersecurity-data-pipeline"
sys.path.insert(0, repo_root)

import importlib
for mod in ['src.ingestion', 'src.cleaning', 'src.transformation', 'src.ml.train', 'src.ml.evaluate']:
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])

from src.ml.evaluate import run_full_evaluation
from src.ml.train import train_models, prepare_training_data

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

spark = SparkSession.builder.appName("j10-evaluation").getOrCreate()
spark.sql("USE CATALOG workspace")

print(f"✓ Spark version: {spark.version}")
print(f"✓ Catalog: workspace")
print(f"✓ Evaluation modules imported")

In [0]:
# === Run the complete ML evaluation pipeline ===
print("=" * 70)
print("ML Evaluation and Investigation")
print("=" * 70)
print("This will:")
print("  1. Reproduce the J9 train/test split")
print("  2. Visualize confusion matrix, ROC, PR curves")
print("  3. Analyze the 6 False Negatives")
print("  4. Check for train/test overlap")
print("  5. Re-evaluate with TEMPORAL split (Mon-Wed train, Thu-Fri test)")
print("  6. Per-day performance analysis (concept drift)")
print("This takes ~5-10 minutes. Be patient.")
print()

results = run_full_evaluation(spark, random_state=42)

print("\n" + "=" * 70)
print("✓ Evaluation completed!")
print("=" * 70)

In [0]:
# === Display all the figures ===
print("--- Figure 1: Confusion Matrix (Stratified Split) ---")
display(results['figures']['cm_stratified'])
plt.show()

print("\n--- Figure 2: ROC Curve ---")
display(results['figures']['roc_curve'])
plt.show()

print("\n--- Figure 3: Precision-Recall Curve ---")
display(results['figures']['pr_curve'])
plt.show()

print("\n--- Figure 4: Confusion Matrix (Temporal Split) ---")
display(results['figures']['cm_temporal'])
plt.show()

In [0]:
# === Comparison: Stratified vs Temporal Split ===
print("=" * 70)
print("CRITICAL COMPARISON: Stratified Split vs Temporal Split")
print("=" * 70)

comparison = pd.DataFrame([
    {
        'Split': 'Stratified (random)',
        'F1-score': results['stratified_metrics']['f1_score'],
        'ROC-AUC': results['stratified_metrics']['roc_auc'],
        'Description': 'Random split, same attack types in train and test',
    },
    {
        'Split': 'Temporal (Mon-Wed / Thu-Fri)',
        'F1-score': results['temporal_metrics']['f1_score'],
        'ROC-AUC': results['temporal_metrics']['roc_auc'],
        'Description': 'Train on first 3 days, test on last 2 days (unseen attacks)',
    },
])

display(comparison)

# Difference
strat_f1 = results['stratified_metrics']['f1_score']
temp_f1 = results['temporal_metrics']['f1_score']
diff = strat_f1 - temp_f1

print(f"\nDifference in F1-score: {diff:.4f}")
if diff < 0.02:
    print("→ Model generalizes well (small degradation)")
elif diff < 0.10:
    print("→ Moderate degradation - some attack types harder to detect")
else:
    print("→ Significant degradation - random split was optimistic")
    print("  The model likely learned attack-type-specific patterns rather than generic attack signatures.")

In [0]:
# === Feature-Identical Observations Check ===
overlap = results['feature_identical_check']

print("=" * 70)
print("Feature-Identical Observations Check")
print("=" * 70)
print(f"\nTrain size:  {overlap['train_size']:,}")
print(f"Test size:   {overlap['test_size']:,}")
print(f"Test observations with feature-identical counterpart in train: {overlap['test_observations_with_train_counterpart']:,}")
print(f"Percentage of test:  {overlap['pct_of_test_with_counterpart']:.2f}%")
print(f"Train observations with counterpart in test: {overlap['train_observations_with_test_counterpart']:,}")
print(f"Percentage of train: {overlap['pct_of_train_with_counterpart']:.2f}%")
print(f"Unique hash overlap (old method, for reference): {overlap['unique_hash_overlap']}")

## Interpretation — Feature-Identical Observations

### What was measured

The number of test observations whose feature vector, after preprocessing and removal of identifying columns (`_source_file`, `timestamp`, `label`, `day_of_week`), has at least one exact counterpart in the training set.

### Caveats

This metric does not automatically indicate data leakage. Two distinct network flows may have identical numerical features once identifying columns are removed. To confirm actual leakage, the original identifying columns (timestamp, source IP) would need to be preserved and compared.

### Status

Further investigation is required, with preserved identifiers, before drawing definitive conclusions regarding data leakage.

In [0]:
# === False Negatives with original attack types ===
print("=" * 70)
print("False Negatives Analysis — Which attacks are missed?")
print("=" * 70)

fn_summary = results['fn_summary']
fn_attack_types = results.get('fn_attack_types', {})

print(f"\n--- Confusion summary ---")
print(f"True Positives  (correctly detected attacks):  {fn_summary['true_positives']:,}")
print(f"False Negatives (MISSED attacks):              {fn_summary['false_negatives']:,}")
print(f"True Negatives  (correctly identified benign): {fn_summary['true_negatives']:,}")
print(f"False Positives (false alarms):                {fn_summary['false_positives']:,}")

if fn_attack_types:
    print(f"\n--- Attack types missed (False Negatives) ---")
    fn_df = pd.DataFrame(
        list(fn_attack_types.items()),
        columns=['attack_type', 'count']
    ).sort_values('count', ascending=False)
    display(fn_df)
else:
    print("\n ! No FN attack types available (label preservation failed).")

In [0]:
# === Attack Type Distribution: train vs test ===
print("=" * 70)
print("Attack Type Distribution: Train (Mon-Wed) vs Test (Thu-Fri)")
print("=" * 70)

attack_dist = results['attack_type_distribution']
df_dist = pd.DataFrame(attack_dist)

display(df_dist)

# Categorize
train_only = df_dist[df_dist['status'] == 'Train only']
test_only = df_dist[df_dist['status'] == 'TEST ONLY (unseen in training)']
both = df_dist[df_dist['status'] == 'Both']

print(f"\nAttack types in BOTH train and test:    {len(both)}")
print(f"Attack types in train only:              {len(train_only)}")
print(f"Attack types in TEST ONLY (unseen):      {len(test_only)}")

if len(test_only) > 0:
    print(f"\n--- UNSEEN attack types in test ---")
    for _, row in test_only.iterrows():
        print(f"  - {row['attack_type']}: {row['test_count']} occurrences in test")

In [0]:
# === False Negatives Analysis ===
print("=" * 70)
print("False Negatives Analysis (Missed Attacks)")
print("=" * 70)

fn = results['fn_summary']
print(f"\nTrue Positives  (correctly detected attacks): {fn['true_positives']:,}")
print(f"False Negatives (MISSED attacks):              {fn['false_negatives']:,}")
print(f"True Negatives  (correctly identified benign): {fn['true_negatives']:,}")
print(f"False Positives (false alarms):                {fn['false_positives']:,}")

print(f"\nRecall = TP / (TP + FN) = {fn['true_positives'] / (fn['true_positives'] + fn['false_negatives']):.4f}")
print(f"Precision = TP / (TP + FP) = {fn['true_positives'] / (fn['true_positives'] + fn['false_positives']):.4f}")

In [0]:
# === Per-day performance analysis ===
per_day = results['per_day_metrics']

print("=" * 70)
print("Per-day Performance Analysis")
print("=" * 70)

if per_day:
    df_per_day = pd.DataFrame(per_day)
    display(df_per_day)
    
    if len(df_per_day) > 1:
        f1_diff = df_per_day['f1'].max() - df_per_day['f1'].min()
        print(f"\nF1 variation across days (Δ): {f1_diff:.3f}")
else:
    print("Per-day analysis not available (day_of_week not preserved in test data).")